In [1]:
from envinit import Workspace, Configuration
config = Configuration()
workspace = Workspace()
workspace.init()

INFO	Workspace initialized successfully.


In [2]:
from transformers import AutoProcessor, Gemma3ForConditionalGeneration
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset, DatasetDict
from PIL import Image
import requests
import torch
import pandas as pd
from collections import Counter
import ast
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_excel("annotated_full.xlsx")
df = df.dropna(subset=["Content"])

In [4]:
def prepare_inputs(df):

    inputs = []
    for row in df.iterrows():
        prompt = row[1]["Prompt"]
        style = ", ".join(ast.literal_eval(row[1]["Style"]))
        persons = ", ".join(ast.literal_eval(row[1]["PER"]))
        organizations = ", ".join(ast.literal_eval(row[1]["ORG"]))
        objects = ", ".join(ast.literal_eval(row[1]["OBJ"]))
        concepts = ", ".join(ast.literal_eval(row[1]["MISC"]))
        title = row[1]["Title"]
        body = row[1]["Content"]

        template = f"""<bos><start_of_turn>user
Sen bir Zaytung haber yazarısın. Aşağıda verilen konu ve bilgiler doğrultusunda yeni bir haberi mizahi bir şekilde oluşturmalısın.

{prompt}

Üslüp: {style}
Kişiler: {persons}
Kurumlar: {organizations}
Objeler: {objects}
Konseptler: {concepts}<end_of_turn>
<start_of_turn>model
Başlık: {title}

Haber: {body}<end_of_turn><eos>"""

        inputs.append(template)

    return inputs

In [5]:
inputs = prepare_inputs(df)

In [6]:
train_val, test = train_test_split(inputs, test_size=0.2, random_state=42)
train, val = train_test_split(train_val, test_size=0.1, random_state=42)

In [7]:
train_dataset = Dataset.from_dict({"text": train})
val_dataset = Dataset.from_dict({"text": val})
test_dataset = Dataset.from_dict({"text": test})

dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

In [8]:
model_name = "google/gemma-3-4b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name, attn_implementation='eager')
model = Gemma3ForConditionalGeneration.from_pretrained(model_name, torch_dtype=torch.bfloat16, attn_implementation='eager')

# Tokenization
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, padding=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Map:   0%|          | 0/2213 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Map:   0%|          | 0/246 [00:00<?, ? examples/s]

Map:   0%|          | 0/615 [00:00<?, ? examples/s]

In [9]:
# Training arguments
training_args = TrainingArguments(
    output_dir="/home/ubuntu/shared/hf_home/hub/gemma-3-4b-ft-2",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    eval_strategy="steps",
    eval_steps=100,               
    logging_steps=100,
    save_steps=100,
    save_total_limit=1,
    num_train_epochs=6,
    learning_rate=2e-5,
    warmup_steps=20,
    lr_scheduler_type="reduce_lr_on_plateau",
    bf16=True,
    optim="adamw_torch_fused",
    weight_decay=0.1,
    report_to="tensorboard",
    logging_dir="./logs",
    load_best_model_at_end=True,
    greater_is_better=False
)



# Enable gradient checkpointing (optional, but helpful for memory)
model.gradient_checkpointing_enable()

# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Training
trainer.train()

# Save final model
trainer.save_model("/home/ubuntu/shared/hf_home/hub/gemma-3-4b-ft-final-2")
tokenizer.save_pretrained("/home/ubuntu/shared/hf_home/hub/gemma-3-4b-ft-final-2")

/tmp/ipykernel_1511302/1617863075.py:31: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
100,2.065700,1.946354
200,1.958600,1.892101
300,1.912700,1.868702
400,1.882400,1.845583
500,1.862500,1.829467
600,1.660900,1.873823
700,1.444600,1.857223
800,1.454200,1.862463
900,1.423800,1.845422
1000,1.439700,1.844717


KeyboardInterrupt: 